[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/fastapi-certified/notebooks/day-05-dependency-injection.ipynb#scrollTo=11223344)

---
# Day 5 · Dependency Injection — Clean, Testable APIs
**certified-journeys / fastapi-certified** · Practice · Day 5 of FastAPI for Python Engineers

> **Goal for today:** Master FastAPI's `Depends()` system to build composable, testable, and DRY route handlers — from simple DB session injection to chained auth dependencies and class-based providers.


In [ ]:
%pip install -q fastapi httpx


## Step 1 · What Is Dependency Injection and Why Does It Matter?

**Dependency Injection (DI)** is the pattern where a function declares what it needs, and a framework provides it — instead of the function creating it internally.

| Without DI | With DI (`Depends`) |
|---|---|
| Route creates DB session inline | Route declares `db = Depends(get_db)` |
| Hard to test — real DB required | Easy to test — swap `get_db` in tests |
| Logic mixed with infrastructure | Logic is pure; infrastructure is injected |
| Repeated setup across routes | One dependency, used everywhere |

FastAPI's DI system is declarative: add `Depends(fn)` as a type annotation or default value in the route signature, and FastAPI calls `fn` before your route runs.


In [ ]:
from fastapi import FastAPI, Depends
from fastapi.testclient import TestClient

app = FastAPI()

# Call log — lets us verify when dependencies are called
call_log: list[str] = []

# A simple dependency — any callable works
def get_settings():
    call_log.append("get_settings called")
    return {"app_name": "MyApp", "debug": False}

# Inject settings into the route via Depends()
@app.get("/info")
def info(settings: dict = Depends(get_settings)):
    # settings is already resolved by FastAPI before this runs
    return {"app": settings["app_name"], "debug": settings["debug"]}

client = TestClient(app)

call_log.clear()
resp = client.get("/info")
print(resp.json())     # {"app": "MyApp", "debug": False}
print(call_log)        # ["get_settings called"] — confirmed per-request


**What just happened?**

- `get_settings` is a plain Python function — no FastAPI imports needed inside it.
- **FastAPI called `get_settings()` automatically** before running `info()` and injected the result.
- If you call `/info` ten times, `get_settings` is called ten times — once per request.
- The route signature is self-documenting: you can read it and know exactly what inputs the handler depends on.


## Step 2 · Database Session Injection with `yield`

The most common use of DI in FastAPI is **database session management**. The `yield` keyword turns a dependency into a context manager:

1. Code before `yield` → runs before the route (setup)
2. `yield` → hands the session to the route
3. Code after `yield` → runs after the route (teardown/cleanup)

This guarantees sessions are **always closed**, even if the route raises an exception.


In [ ]:
from typing import Generator

# Simulated "database session" — in production this would be SQLAlchemy Session
class FakeDBSession:
    def __init__(self, connection_id: int):
        self.connection_id = connection_id
        self.closed = False

    def query(self, table: str) -> list:
        return [{"table": table, "conn": self.connection_id}]

    def close(self):
        self.closed = True

session_counter = 0

# Generator dependency — yield turns it into a context manager
def get_db() -> Generator:
    global session_counter
    session_counter += 1
    db = FakeDBSession(connection_id=session_counter)
    try:
        yield db        # FastAPI injects db into the route
    finally:
        db.close()      # Always runs — even on exceptions

app2 = FastAPI()

@app2.get("/users")
def list_users(db: FakeDBSession = Depends(get_db)):
    rows = db.query("users")  # db is open here
    return rows

client2 = TestClient(app2)

resp = client2.get("/users")
print(resp.json())  # [{"table": "users", "conn": 1}]

# Make a second request — new session each time
resp2 = client2.get("/users")
print(resp2.json())  # [{"table": "users", "conn": 2}]

print(f"Total sessions created: {session_counter}")  # 2


**What just happened?**

- Each request got a **fresh, independent database session** — `conn` increments.
- The `finally` block ensures `db.close()` runs even if the route throws an exception.
- **In production** you'd replace `FakeDBSession` with a real SQLAlchemy `SessionLocal()` — the dependency pattern stays identical.
- To test without a real DB: `app.dependency_overrides[get_db] = lambda: FakeDBSession(0)` — no code changes needed.


## Step 3 · Query Parameter Dependencies — Reusable Pagination

Dependencies aren't limited to infrastructure. They're equally useful for **shared query-parameter groups** like pagination, filtering, or sorting.

Instead of repeating `skip: int = 0, limit: int = 20` across dozens of routes, define it once as a dependency and inject it everywhere.


In [ ]:
from fastapi import FastAPI, Depends, Query

# Pagination dependency — FastAPI reads skip/limit from query params
def pagination(skip: int = Query(0, ge=0), limit: int = Query(20, ge=1, le=100)):
    return {"skip": skip, "limit": limit}

app3 = FastAPI()

ITEMS = [f"item-{i}" for i in range(50)]
USERS = [f"user-{i}" for i in range(30)]

# Both routes share identical pagination — DRY
@app3.get("/items")
def list_items(page: dict = Depends(pagination)):
    return ITEMS[page["skip"]: page["skip"] + page["limit"]]

@app3.get("/users")
def list_users(page: dict = Depends(pagination)):
    return USERS[page["skip"]: page["skip"] + page["limit"]]

client3 = TestClient(app3)

# Default pagination
resp1 = client3.get("/items")
print("Default (20 items):", len(resp1.json()), resp1.json()[:3])

# Custom pagination
resp2 = client3.get("/items", params={"skip": 10, "limit": 5})
print("Skip 10, limit 5:", resp2.json())

# Validation: limit > 100 is rejected
resp3 = client3.get("/items", params={"limit": 200})
print("Limit 200 status:", resp3.status_code)  # 422


**What just happened?**

- The `pagination` dependency reads `skip` and `limit` from the query string, **with built-in validation** (`ge=0`, `le=100`).
- Both `/items` and `/users` share the same pagination logic without any code duplication.
- **Invalid values (limit=200) return 422** automatically — Pydantic validation is applied inside the dependency.
- OpenAPI docs will show `skip` and `limit` as query parameters on every route that injects `pagination`.


## Step 4 · Auth Dependency — Reading the Authorization Header

A common pattern is extracting and validating a bearer token from the `Authorization` header. The dependency:
1. Reads the header
2. Validates the format
3. Returns the token (or raises 401)

Routes that need auth simply declare `token: str = Depends(get_current_user)`.


In [ ]:
from fastapi import FastAPI, Depends, HTTPException, Header
from typing import Optional

# Simulated token → user mapping (in production: JWT decode or DB lookup)
FAKE_TOKENS: dict[str, dict] = {
    "token-alice": {"id": 1, "username": "alice", "active": True},
    "token-bob":   {"id": 2, "username": "bob",   "active": False},  # inactive
}

def get_current_user(authorization: Optional[str] = Header(None)):
    """Extract bearer token and look up the user."""
    if not authorization:
        raise HTTPException(status_code=401, detail="Authorization header missing",
                            headers={"WWW-Authenticate": "Bearer"})
    if not authorization.startswith("Bearer "):
        raise HTTPException(status_code=401, detail="Invalid Authorization format")

    token = authorization.split(" ", 1)[1]  # strip "Bearer "
    user = FAKE_TOKENS.get(token)
    if not user:
        raise HTTPException(status_code=401, detail="Invalid or expired token")
    return user

app4 = FastAPI()

@app4.get("/me")
def get_me(user: dict = Depends(get_current_user)):
    return {"id": user["id"], "username": user["username"]}

client4 = TestClient(app4)

# Valid token
r_ok = client4.get("/me", headers={"Authorization": "Bearer token-alice"})
print("Valid:", r_ok.status_code, r_ok.json())

# No header
r_no = client4.get("/me")
print("No header:", r_no.status_code, r_no.json())

# Wrong format
r_fmt = client4.get("/me", headers={"Authorization": "token-alice"})
print("Bad format:", r_fmt.status_code, r_fmt.json())

# Unknown token
r_bad = client4.get("/me", headers={"Authorization": "Bearer token-unknown"})
print("Unknown:", r_bad.status_code, r_bad.json())


**What just happened?**

- `Header(None)` tells FastAPI to read the `Authorization` HTTP header (FastAPI normalizes header names to lowercase).
- The dependency raises `HTTPException(401)` in all error paths — the route never runs.
- **If the dependency raises, FastAPI short-circuits** — the route handler function is never called.
- Every route that needs auth gets a clean, typed `user: dict` with zero repeated logic.


## Step 5 · Sub-dependencies — Chaining `Depends`

Dependencies can themselves depend on other dependencies. FastAPI resolves the full chain automatically.

```
get_current_active_user
    └── get_current_user
            └── Authorization header
```

FastAPI caches dependency results **within a single request** by default — so `get_current_user` runs only once even if both `get_current_active_user` and another dependency call it.


In [ ]:
from fastapi import FastAPI, Depends, HTTPException, Header
from typing import Optional

FAKE_TOKENS2: dict[str, dict] = {
    "token-alice": {"id": 1, "username": "alice", "active": True},
    "token-bob":   {"id": 2, "username": "bob",   "active": False},
}

call_counts: dict[str, int] = {"get_current_user": 0, "get_current_active_user": 0}

def get_current_user2(authorization: Optional[str] = Header(None)):
    call_counts["get_current_user"] += 1
    if not authorization or not authorization.startswith("Bearer "):
        raise HTTPException(status_code=401, detail="Unauthorized")
    token = authorization.split(" ", 1)[1]
    user = FAKE_TOKENS2.get(token)
    if not user:
        raise HTTPException(status_code=401, detail="Invalid token")
    return user

# Sub-dependency: depends on get_current_user2
def get_current_active_user(user: dict = Depends(get_current_user2)):
    call_counts["get_current_active_user"] += 1
    if not user["active"]:
        raise HTTPException(status_code=403, detail="Inactive user account")
    return user

app5 = FastAPI()

# Route requires an active user
@app5.get("/dashboard")
def dashboard(user: dict = Depends(get_current_active_user)):
    return {"message": f"Welcome, {user['username']}!"}

# Route only requires any authenticated user (inactive allowed)
@app5.get("/profile")
def profile(user: dict = Depends(get_current_user2)):
    return {"username": user["username"], "active": user["active"]}

client5 = TestClient(app5)

# Active user → dashboard OK
r1 = client5.get("/dashboard", headers={"Authorization": "Bearer token-alice"})
print("Active user dashboard:", r1.status_code, r1.json())

# Inactive user → 403
r2 = client5.get("/dashboard", headers={"Authorization": "Bearer token-bob"})
print("Inactive user dashboard:", r2.status_code, r2.json())

# Inactive user → profile OK (less restrictive)
r3 = client5.get("/profile", headers={"Authorization": "Bearer token-bob"})
print("Inactive user profile:", r3.status_code, r3.json())

print("Dependency call counts:", call_counts)


**What just happened?**

- `get_current_active_user` **chains into `get_current_user2`** — the auth check runs automatically.
- `alice` (active=True) gets into `/dashboard`; `bob` (active=False) is rejected at the sub-dependency level.
- `/profile` uses only `get_current_user2` — so `bob` can still view their profile.
- **FastAPI's per-request caching** means `get_current_user2` runs only once per request even if multiple dependencies need it.


## Step 6 · Class-Based Dependencies

When a dependency needs configuration (e.g., different query limits per route), a **class** is cleaner than a closure. FastAPI detects that the class is callable (`__init__` + `__call__`) and treats it as a dependency.

The class instance is created when the route is defined; `__call__` is invoked per request.

```python
class Paginator:
    def __init__(self, max_limit: int = 100):
        self.max_limit = max_limit
    def __call__(self, skip: int = 0, limit: int = 20) -> dict:
        ...
```


In [ ]:
from fastapi import FastAPI, Depends, Query

class Paginator:
    """Configurable pagination dependency."""

    def __init__(self, max_limit: int = 100):
        # Configured at route-definition time, not per-request
        self.max_limit = max_limit

    def __call__(self, skip: int = Query(0, ge=0),
                 limit: int = Query(20, ge=1)) -> dict:
        # Called per-request — limit is clamped to max_limit
        limit = min(limit, self.max_limit)
        return {"skip": skip, "limit": limit}

# Different max limits per resource
items_paginator = Paginator(max_limit=50)
logs_paginator  = Paginator(max_limit=10)  # logs are expensive — tighter cap

app6 = FastAPI()

ITEMS2 = list(range(200))
LOGS   = list(range(500))

@app6.get("/items")
def list_items(page: dict = Depends(items_paginator)):
    return {"data": ITEMS2[page["skip"]: page["skip"] + page["limit"]], **page}

@app6.get("/logs")
def list_logs(page: dict = Depends(logs_paginator)):
    return {"data": LOGS[page["skip"]: page["skip"] + page["limit"]], **page}

client6 = TestClient(app6)

# Items: limit 200 → capped at 50
r_items = client6.get("/items", params={"limit": 200})
print("Items limit:", r_items.json()["limit"])  # 50

# Logs: limit 200 → capped at 10
r_logs = client6.get("/logs", params={"limit": 200})
print("Logs limit: ", r_logs.json()["limit"])   # 10


**What just happened?**

- `Paginator(max_limit=50)` creates a **configured instance** at route-definition time.
- **`__call__` is invoked per request** — FastAPI calls it just like a function dependency.
- The same class provides different behavior (`max_limit=50` vs `max_limit=10`) via configuration, not code duplication.
- Class-based dependencies are ideal when you need **stateful configuration** or when a dependency group grows complex enough to deserve its own class.


## Step 7 · Overriding Dependencies in Tests

This is FastAPI's **killer feature for testing**. `app.dependency_overrides` lets you swap any dependency for a test stub without changing production code.

```python
app.dependency_overrides[get_db] = lambda: fake_db
```

After the test, clear overrides to restore production behavior:
```python
app.dependency_overrides = {}
```


In [ ]:
from fastapi import FastAPI, Depends

# Production dependency — expensive (e.g., real DB connection)
class ProductionDB:
    def get_user(self, user_id: int):
        # In production, hits a real database
        return {"id": user_id, "name": "PRODUCTION_USER"}

def get_db_production():
    yield ProductionDB()

app7 = FastAPI()

@app7.get("/users/{user_id}")
def get_user(user_id: int, db=Depends(get_db_production)):
    return db.get_user(user_id)

# --- Testing with dependency_overrides ---
class TestDB:
    def get_user(self, user_id: int):
        return {"id": user_id, "name": "TEST_USER"}  # controlled test data

def get_db_test():
    yield TestDB()

# Swap the dependency — no production code changes needed
app7.dependency_overrides[get_db_production] = get_db_test

client7 = TestClient(app7)

resp = client7.get("/users/42")
print("With test override:", resp.json())  # TEST_USER — not PRODUCTION_USER

# Restore production behavior
app7.dependency_overrides = {}

print("Override cleared:", app7.dependency_overrides)  # {}


**What just happened?**

- `app.dependency_overrides[get_db_production] = get_db_test` **swaps the dependency globally** for that `app` instance.
- The route (`get_user`) was not modified at all — the override happens at the framework level.
- **In a pytest `conftest.py`**, you'd define a `client` fixture that sets overrides and tears them down, giving every test a clean isolated state.
- This is why DI makes FastAPI apps dramatically easier to test than apps with hardcoded infrastructure calls.


## Step 8 · Router-Level and App-Level Dependencies

You don't have to inject dependencies on every route. FastAPI lets you attach them at two higher levels:

- **`APIRouter(dependencies=[...])`** — applies to all routes in that router
- **`FastAPI(dependencies=[...])`** — applies to every route in the entire app

This is ideal for cross-cutting concerns like authentication, rate limiting, or logging.


In [ ]:
from fastapi import FastAPI, APIRouter, Depends, HTTPException, Header
from typing import Optional

API_KEY = "admin-key-xyz"

def verify_api_key(x_api_key: Optional[str] = Header(None)):
    """Applied to all routes in the admin router."""
    if x_api_key != API_KEY:
        raise HTTPException(status_code=403, detail="Invalid API key")

# Router with a blanket dependency — all routes inside are protected
admin_router = APIRouter(
    prefix="/admin",
    dependencies=[Depends(verify_api_key)],  # applied automatically
)

@admin_router.get("/stats")
def admin_stats():
    return {"users": 42, "uptime": "99.9%"}  # no Depends here — inherited

@admin_router.get("/config")
def admin_config():
    return {"debug": False, "version": "1.0.0"}

app8 = FastAPI()
app8.include_router(admin_router)

# Public route — not protected
@app8.get("/health")
def health():
    return {"status": "ok"}

client8 = TestClient(app8)

# Public route — always accessible
print("Health:", client8.get("/health").status_code)  # 200

# Admin route — no key
print("Admin no key:", client8.get("/admin/stats").status_code)  # 403

# Admin route — valid key
r_ok = client8.get("/admin/stats", headers={"x-api-key": API_KEY})
print("Admin with key:", r_ok.status_code, r_ok.json())

# Admin config — also protected by the same router-level dep
r_cfg = client8.get("/admin/config", headers={"x-api-key": API_KEY})
print("Admin config:", r_cfg.status_code, r_cfg.json())


**What just happened?**

- `APIRouter(dependencies=[Depends(verify_api_key)])` protects **all routes** in that router without touching individual route definitions.
- `/health` is on the main `app8`, not the admin router — so it's unprotected.
- **`app.include_router(router)`** merges the router's routes (and their dependencies) into the app.
- Router-level dependencies are the idiomatic way to apply middleware-like logic (auth, logging, rate limiting) to a subset of routes.


In [ ]:
# Challenge: Build a complete DI chain
#
# Create an API with three layers of dependency:
#   1. get_db() → yields a FakeDB with a query(table) method
#   2. get_current_user(authorization=Header(None), db=Depends(get_db))
#      → reads token from header, looks user up in db
#   3. get_admin_user(user=Depends(get_current_user))
#      → raises 403 if user["role"] != "admin"
#
# Routes:
#   GET /profile  → requires any authenticated user (Depends(get_current_user))
#   GET /admin    → requires admin role (Depends(get_admin_user))
#
# Test with TestClient:
#   - Valid regular user: /profile 200, /admin 403
#   - Valid admin user:   /profile 200, /admin 200
#   - No auth header:     /profile 401
#
# Bonus: use dependency_overrides to test without a real token

from fastapi import FastAPI, Depends, HTTPException, Header
from fastapi.testclient import TestClient
from typing import Optional, Generator

# Your solution here
# class FakeDB: ...
# def get_db() -> Generator: ...
# def get_current_user(...): ...
# def get_admin_user(...): ...
# app = FastAPI()
# @app.get("/profile") ...
# @app.get("/admin") ...
# client = TestClient(app)
# ...


---
## Day 5 key concepts recap

| Concept | What to remember |
|---|---|
| `Depends(fn)` | FastAPI resolves `fn` before the route runs; result is injected |
| `yield` dependency | Code before `yield` = setup; code after = teardown (always runs) |
| Sub-dependencies | Dependencies chain automatically; results are cached per request |
| Class-based deps | Use `__call__` for per-request logic with `__init__`-time config |
| `dependency_overrides` | Swap any dep in tests without changing production code |
| Router-level deps | `APIRouter(dependencies=[...])` protects all routes in that router |
| Query param deps | Reuse validation groups (pagination, filtering) across routes |

> **Tip:** Dependency injection is FastAPI's superpower for testability. Override any dependency in tests with `app.dependency_overrides[get_db] = get_test_db`.

---
## What's next
**Day 6** → CRUD API with In-Memory Storage — wire everything together into a complete, production-shaped Items API with pagination and `APIRouter`.

Mark Day 5 complete in your [tracker](../index.html).
